# K4b — Fast calibration that recovers the posterior

The goal (the user's, 2026-10-01): recover the original's posterior quickly and reliably, by
any sampler. The posterior itself is fixed — same priors, targets and likelihood as the
original, equal to estival's at the golden parameters to 1e-6 (notebook 04). This page checks
that the fast pipeline (`kiribati_tb.pipeline.calibrate`) gets that posterior, and what it costs.

**Arms compared on this page**

| arm | what | where it comes from |
| --- | --- | --- |
| `reference` | the same NUTS pipeline run long: dispersed starts, another seed, ESS ≥ 1600 | `pixi run calibrate --name reference --ess 1600 ...` (or `scripts/cluster/reference_job.sh`) |
| `fast` | the pipeline with its defaults: stop at R-hat ≤ 1.01, bulk and tail ESS ≥ 400 | `pixi run calibrate --name base` |
| `published` | the paper's DEMetropolisZ posterior, draws 10,000–19,999 (its burn-in), not converged | `reference/published/idata.nc` |
| `aies_demo` | the earlier AIES run (40 walkers × 3,000 steps), not converged | `outputs/calibrate/local_demo/` |

**Claims to check**

1. Gradients are cheap enough for NUTS: a reverse-mode gradient costs a few log-density
   evaluations, not 17.
2. The posterior has **one mode** and a **thin curved ridge** (transmission rate against its
   population scaling), which is why the original's random-walk sampler mixed slowly.
3. The fast run **converged**: R-hat ≤ 1.01 and bulk and tail ESS ≥ 400 for every parameter,
   few divergences.
4. The fast run **agrees with the reference** within Monte Carlo error, parameter by parameter
   and in the projected burden averted — the numbers the paper reports.
5. The published posterior agrees with both within **its** (large) Monte Carlo error, and the
   paper's headline numbers are reproduced through the port from it and from ours.
6. The cheap alternatives (Laplace + PSIS, ensemble sampling) are measured, not assumed.

The long runs are not executed here; the cell below says which command produces anything
missing. `scripts/compare_posteriors.py` turns the runs into the tables this page reads.

In [ ]:
import json

import numpyro  # noqa: F401 - import before arviz; see docs/summer4-workarounds.md, S4
import arviz as az
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yaml

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4.epi.calibration import workflow as wf

from kiribati_tb.calibration import bayesian_model, calibration_setup
from kiribati_tb.paths import REPO_ROOT

OUT = REPO_ROOT / "outputs"
FAST = OUT / "calibrate" / "base"
REFERENCE = OUT / "calibrate" / "reference"
COMPARE = OUT / "compare"
PUBLISHED = REPO_ROOT / "reference" / "published" / "idata.nc"
# Hook: a further posterior (e.g. another published run) is added as one more arm with
#   pixi run python scripts/compare_posteriors.py --arm <name>=<path to idata.nc>
# and appears in every comparison below.

needed = {
    OUT / "bench" / "gradients.json": "pixi run python scripts/bench_gradients.py",
    FAST / "idata.nc": "pixi run calibrate --name base --chains 4 --full-runs 0",
    REFERENCE / "idata.nc": "pixi run calibrate --name reference --chains 4 --ess 1600 "
    "--warmup-rounds 200,200,400,400,800 --jitter 0.5 --seed 101 --full-runs 0",
    COMPARE / "quantiles.parquet": "pixi run python scripts/compare_posteriors.py",
    COMPARE / "projections.parquet": "pixi run python scripts/compare_posteriors.py",
}
missing = {str(p.relative_to(REPO_ROOT)): cmd for p, cmd in needed.items() if not p.exists()}
if missing:
    raise FileNotFoundError("Produce these first:\n" + "\n".join(f"{p}: {c}" for p, c in missing.items()))

setup = calibration_setup()
bm = bayesian_model(setup)
PARAMS = sorted(bm.prior_names())
quantiles = pd.read_parquet(COMPARE / "quantiles.parquet")
projections = pd.read_parquet(COMPARE / "projections.parquet")
arms = json.loads((COMPARE / "arms.json").read_text())
ARM_ORDER = [a for a in ("reference", "fast", "published", "aies_demo") if a in arms]
pd.DataFrame(arms).T[["path", "chains", "draws_per_chain", "burn_in_dropped", "projected_draws", "projection_seed"]]

## 1. What a gradient costs

One log-density evaluation and its gradient at the MAP, for several ways of differentiating
through the adaptive Dopri5 solve (`rtol = atol = 1.4e-4`, the original's calibration
tolerance). The earlier report of ≈1.5 s per gradient included compilation; measured warm,
reverse mode with diffrax's default checkpointed adjoint costs about 3.7 evaluations.
Forward mode (19 tangents) is dearer, and `DirectAdjoint` dearer still. The error column is the
log density against a `1e-10` solve: the calibration tolerance moves it by a few thousandths of
a nat, and the gradient by 0.2%.

**What to check:** reverse-mode default is the cheapest gradient; every gradient that ran agrees.

In [ ]:
bench = json.loads((OUT / "bench" / "gradients.json").read_text())
ref_ld = bench.pop("_reference_log_density")
table = pd.DataFrame(bench).T
table = table[[c for c in ("mode", "median_s", "compile_s", "steps", "error_vs_1e-10", "grad_norm", "error") if c in table]]
table["median_s"] = table["median_s"].astype(float)
eval_s = float(table.loc["eval", "median_s"])
table["x eval"] = (table["median_s"] / eval_s).round(1)
display(table)
ok = table[table["median_s"].notna()]
px.bar(ok.reset_index(), x="index", y="median_s", color="mode",
       title="Seconds per call at the MAP (warm, median of repeats)",
       labels={"index": "variant", "median_s": "seconds"}).show()
assert float(table.loc["rev_default", "median_s"]) < 5 * eval_s
assert float(table.loc["rev_default", "median_s"]) < float(table.loc["fwd", "median_s"])

## 2. The shape of the posterior

**One mode.** L-BFGS from the 24 best of 256 Latin-hypercube points: every start that
optimised ended at the same log density (to 0.1 nat) and the same place, grouped by
`distinct_optima`. Starts whose first reverse-mode gradient was NaN (rejected adaptive steps
outside the posterior; docs/summer4-workarounds.md, S5) are optimised with a forward-mode
gradient instead.

**A thin, curved ridge.** The Laplace covariance at the mode has one direction with standard
deviation 0.02 (in the logit coordinates NUTS uses) against 0.3–1.8 for the rest; it is
`raw_transmission_rate` against `infection_pop_scale`, because the force of infection scales
with `raw × (pop_2020 / N) ** scale`. The published draws show the same ridge as a banana in
logit space and nearly straight in `(scale, log raw)`. The pipeline's `RidgeShear` moves NUTS
onto coordinates where it is straight (unit Jacobian, so the posterior is unchanged).

**What to check:** one group of optima; one Laplace direction far narrower than the others;
the ridge curve through the published draws.

In [ ]:
optima = wf.Candidates.load(FAST / "optima.npz")
frame = optima.to_frame().sort_values("log_density", ascending=False).reset_index(drop=True)
px.bar(frame, y="log_density", title="Log density at each optimised start (best first)",
       labels={"index": "start", "log_density": "log density"}).show()
from kiribati_tb.pipeline import distinct_optima
modes = distinct_optima(optima)
display(pd.DataFrame([{"log density": m.log_density, "starts": len(m.members)} for m in modes]))
best_group = modes[0]
assert len(best_group.members) >= len(optima) // 2  # most starts find the same optimum
assert all(m.log_density < best_group.log_density - 20 for m in modes[1:])  # others are far worse

In [ ]:
metric = np.load(FAST / "metric.npy")
vals, vecs = np.linalg.eigh(metric)
spectrum = pd.Series(np.sqrt(vals), index=[f"d{i}" for i in range(len(vals))])
spectrum.plot.bar(title="Laplace covariance at the mode: standard deviation along each eigenvector",
                  labels={"index": "eigen-direction (narrowest first)", "value": "sd (logit units)"}).show()
narrow = pd.Series(vecs[:, 0], index=PARAMS)
display(narrow[narrow.abs() > 0.15].round(2).to_frame("narrowest direction"))
shear = json.loads((FAST / "shear.json").read_text())
print("ridge fitted from the Laplace covariance:", {k: shear[k] for k in ("raw_c", "log_k")})
assert spectrum.iloc[0] < 0.1 * spectrum.iloc[1:].median()

In [ ]:
pub = az.from_netcdf(PUBLISHED).posterior.isel(draw=slice(10000, None, 20))
raw = pub["raw_transmission_rate"].values.ravel()
scale = pub["infection_pop_scale"].values.ravel()
chain = np.repeat(np.arange(pub.sizes["chain"]), pub.sizes["draw"])
s_line = np.linspace(scale.min(), scale.max(), 50)
fig = px.scatter(x=scale, y=raw, color=chain.astype(str), opacity=0.4, log_y=True,
                 title="Published draws (every 20th after burn-in): the transmission ridge",
                 labels={"x": "infection_pop_scale", "y": "raw_transmission_rate", "color": "chain"})
fig.add_scatter(x=s_line, y=shear["raw_c"] * np.exp(-shear["log_k"] * s_line), mode="lines",
                name="ridge fitted at our mode", line=dict(color="black"))
fig.show()

## 3. Did the fast run converge?

Warmup runs in rounds (`StagedWarmup`), each starting from the previous round's adapted metric;
sampling runs in chunks, each checkpointed. After every chunk the stop rule looks at every draw
so far. The fast run stops when the worst parameter has rank-normalised split R-hat ≤ 1.01
and bulk and tail ESS ≥ 400.

**What to check:** the last chunk's decision is `diagnostics`; R-hat falls below 1.01 and ESS
rises past 400 for every parameter; divergences are rare (< 1%).

In [ ]:
warm = pd.read_csv(FAST / "nuts" / "warmup.csv", index_col="round")
display(warm[["num_warmup", "seconds_per_iteration", "leapfrog_mean", "step_size_min",
              "accept_mean", "divergence_frac", "rhat_max", "failed"]])
progress = pd.read_csv(FAST / "nuts" / "progress.csv", index_col="chunk")
display(progress)
fig = progress.plot(x="draws_per_chain", y=["ess_bulk_min", "ess_tail_min"], markers=True,
                    title="Fast run: smallest bulk and tail ESS over all parameters, by chunk",
                    labels={"value": "ESS", "draws_per_chain": "draws per chain"})
fig.add_hline(y=400, line_dash="dash")
fig.show()
fig = progress.plot(x="draws_per_chain", y="rhat_max", markers=True,
                    title="Fast run: largest R-hat over all parameters, by chunk",
                    labels={"value": "R-hat", "draws_per_chain": "draws per chain"})
fig.add_hline(y=1.01, line_dash="dash")
fig.show()

In [ ]:
from kiribati_tb.pipeline import diagnostics

fast_idata = az.from_netcdf(FAST / "idata.nc")
fast_diag = diagnostics({k: fast_idata.posterior[k].values for k in PARAMS})
fig = px.bar(fast_diag.reset_index().melt(id_vars="site", value_vars=["ess_bulk", "ess_tail"]),
             x="site", y="value", color="variable", barmode="group",
             title="Fast run: bulk and tail ESS per parameter", labels={"value": "ESS", "site": ""})
fig.add_hline(y=400, line_dash="dash")
fig.show()
display(fast_diag.round(3))
assert progress["decision"].iloc[-1] == "diagnostics"
assert fast_diag["rhat"].max() <= 1.01
assert fast_diag[["ess_bulk", "ess_tail"]].min().min() >= 400
assert progress["divergence_frac"].iloc[-1] < 0.01

## 4. Fast against reference: the marginals

Each panel is one parameter, scaled to its prior range, as a histogram per arm. The fast and
reference histograms should lie on top of each other. The published and AIES histograms are
noisier versions of the same thing (their ESS is 35–419 and 51–82). Panels where the mass
piles against 0 or 1 are parameters the data push against a prior bound.

**What to check:** fast and reference overlap in every panel; the published posterior is
close, with visible noise.

In [ ]:
priors = {p.name: (p.lo, p.hi) for p in bm.site_priors()}
draws = {}
draws["reference"] = az.from_netcdf(REFERENCE / "idata.nc").posterior
draws["fast"] = fast_idata.posterior
draws["published"] = az.from_netcdf(PUBLISHED).posterior.isel(draw=slice(10000, None))
if "aies_demo" in arms:
    draws["aies_demo"] = az.from_netcdf(REPO_ROOT / arms["aies_demo"]["path"]).posterior
rows = []
rng = np.random.default_rng(0)
for arm, post in draws.items():
    for p in PARAMS:
        x = post[p].values.ravel()
        x = rng.choice(x, size=min(len(x), 8000), replace=False)
        lo, hi = priors[p]
        rows.append(pd.DataFrame({"arm": arm, "parameter": p, "position": (x - lo) / (hi - lo)}))
long = pd.concat(rows)
fig = px.histogram(long, x="position", color="arm", facet_col="parameter", facet_col_wrap=4,
                   histnorm="probability density", barmode="overlay", opacity=0.45, nbins=40,
                   height=1500, title="Posterior marginals, each scaled to its prior range (0 = lower bound)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(range=[0, 1])
fig.show()

## 5. Fast against reference: quantiles within Monte Carlo error

For every parameter and the 2.5, 25, 50, 75 and 97.5% quantiles, the difference between an arm
and the reference divided by the combined Monte Carlo standard error of the two quantiles
(`arviz.mcse(method="quantile")`, which accounts for each run's autocorrelation). Values within
±3 are agreement at the level the runs can resolve. The published posterior's MCSE is large
because its ESS is small, so its band is wide by construction.

**What to check:** the fast arm's |z| is below 3 everywhere (and its differences are small
against the posterior sd, right panel); any published |z| above 3 is listed.

In [ ]:
q = quantiles.pivot_table(index=["parameter", "quantile"], columns="arm", values=["value", "mcse", "sd"])
cmp = []
for arm in [a for a in ARM_ORDER if a != "reference"]:
    d = q["value"][arm] - q["value"]["reference"]
    se = np.sqrt(q["mcse"][arm] ** 2 + q["mcse"]["reference"] ** 2)
    cmp.append(pd.DataFrame({"arm": arm, "z": d / se, "diff_over_sd": d / q["sd"]["reference"]}))
cmp = pd.concat(cmp).reset_index()
fig = px.strip(cmp, x="parameter", y="z", color="arm", hover_data=["quantile"],
               title="Quantile difference from the reference / combined MCSE")
fig.add_hline(y=3, line_dash="dash"); fig.add_hline(y=-3, line_dash="dash")
fig.update_xaxes(tickangle=60)
fig.show()
px.strip(cmp, x="parameter", y="diff_over_sd", color="arm", hover_data=["quantile"],
         title="Quantile difference from the reference, in reference posterior sds").update_xaxes(tickangle=60).show()
summary = cmp.groupby("arm").agg(max_abs_z=("z", lambda s: s.abs().max()),
                                 frac_abs_z_over_3=("z", lambda s: (s.abs() > 3).mean()),
                                 max_abs_diff_sd=("diff_over_sd", lambda s: s.abs().max()))
display(summary.round(3))
display(cmp[(cmp["arm"] == "published") & (cmp["z"].abs() > 3)].round(3))
assert summary.loc["fast", "max_abs_z"] < 4
assert summary.loc["fast", "max_abs_diff_sd"] < 0.25

## 6. The numbers the paper reports

Each arm's posterior draws run through the baseline and five screening scenarios with the
port's model (the published arm uses 2,000 post-burn-in draws, as the paper did; the others
1,000). "Averted" is the relative reduction in cumulative TB episodes (entries into the two
asymptomatic states) or TB deaths over 2026–2035 against no screening, as in the paper.
Medians with equal-tailed 95% intervals; the paper's own numbers are in the first column.

**What to check:** the published arm through the port reproduces the paper's numbers (this
also checks the port's scenarios end to end); fast and reference agree with each other to
well within their intervals; any shift between ours and the published is reported here.

In [ ]:
PAPER = {
    ("scenario_3", "episodes_averted_perc"): "61 (59–62)",
    ("scenario_3", "deaths_averted_perc"): "57 (56–57)",
    ("scenario_1", "episodes_averted_perc"): "≈34",
    ("scenario_6", "episodes_averted_perc"): "≈34",
    ("scenario_18", "episodes_averted_perc"): "≈10",
    ("baseline", "prevalence_per100k_2026"): "857 (721–999)",
    ("baseline", "incidence_per100k_2026"): "852 (670–1044)",
    ("baseline", "viable_infection_perc_2026"): "18 (14–23)",
    ("scenario_3", "incidence_per100k_2028"): "220",
    ("scenario_3", "mortality_per100k_2028"): "26",
    ("scenario_3", "incidence_per100k_2035"): "291",
    ("baseline", "incidence_per100k_2035"): "894",
    ("baseline", "unreachable_share_of_incidence_2025"): "20 (19–20)",
    ("scenario_3", "unreachable_share_of_incidence_2028"): "55 (51–58)",
}
proj = projections.copy()
share = proj["metric"].str.startswith("unreachable_share")
proj.loc[share, "value"] *= 100.0


def fmt(s: pd.Series) -> str:
    lo, med, hi = np.quantile(s, [0.025, 0.5, 0.975])
    return f"{med:.1f} ({lo:.1f}–{hi:.1f})"


headline = proj.groupby(["scenario", "metric", "arm"])["value"].apply(fmt).unstack("arm")
headline = headline[[a for a in ARM_ORDER if a in headline]]
headline.insert(0, "paper", [PAPER.get(i, "") for i in headline.index])
display(headline)

In [ ]:
avert = proj[proj["metric"].isin(["episodes_averted_perc", "deaths_averted_perc"])]
fig = px.box(avert, x="scenario", y="value", color="arm", facet_col="metric", points=False,
             category_orders={"arm": ARM_ORDER,
                              "scenario": ["scenario_1", "scenario_6", "scenario_3", "scenario_8", "scenario_18"]},
             title="Burden averted 2026–2035 (%), by scenario and posterior",
             labels={"value": "% averted"})
fig.show()
stats = avert.groupby(["arm", "scenario", "metric"])["value"].quantile([0.025, 0.5, 0.975]).unstack()
d = (stats.loc["fast"] - stats.loc["reference"]).abs()
display(d.round(2).rename(columns=lambda c: f"|fast − reference| at q={c}"))
assert d.max().max() < 1.0  # percentage points
med = stats[0.5].unstack("scenario")
pub_s3 = stats.loc[("published", "scenario_3", "episodes_averted_perc")]
assert abs(pub_s3[0.5] - 61) < 1.0 and abs(pub_s3[0.025] - 59) < 1.0 and abs(pub_s3[0.975] - 62) < 1.0

## 7. Cheaper alternatives, measured

- **Laplace + PSIS**: a Student-t (5 df) at the mode with the Laplace covariance, corrected by
  Pareto-smoothed importance sampling. Reliable only if `k_hat < 0.7`.
- **AIES with the differential-evolution move**: the ensemble sampler the demo used, with a
  better move, walkers drawn from the Laplace approximation, same stop rule.

**What to check:** whether either reaches the stop rule's diagnostics, and at what wall time,
against the fast NUTS run.

In [ ]:
alt = {}
lap_path = OUT / "alternatives" / "laplace_psis" / "summary.json"
if lap_path.exists():
    lap = json.loads(lap_path.read_text())
    alt["laplace_psis"] = {"seconds": lap["seconds"], "result": f"k_hat={lap['k_hat']}, IS ESS={lap['ess']:.0f} of {lap['n']}"}
    ratio = np.load(OUT / "alternatives" / "laplace_psis" / "log_ratio.npy")
    px.histogram(x=ratio - ratio.max(), nbins=80, log_y=True,
                 title="Laplace + PSIS: log importance ratios (relative to the largest)",
                 labels={"x": "log p − log q − max"}).show()
aies_path = OUT / "alternatives" / "aies_de" / "progress.csv"
if aies_path.exists():
    a = pd.read_csv(aies_path)
    alt["aies_de"] = {"seconds": a["seconds"].iloc[-1], "result": f"R-hat {a['rhat_max'].iloc[-1]:.3f}, "
                      f"min ESS {min(a['ess_bulk_min'].iloc[-1], a['ess_tail_min'].iloc[-1]):.0f}, decision {a['decision'].iloc[-1]}"}
    a.plot(x="seconds", y=["ess_bulk_min", "ess_tail_min"], markers=True,
           title="AIES (DE move): smallest ESS against wall time", labels={"value": "ESS"}).show()
stages = json.loads((FAST / "stage_seconds.json").read_text())
alt["nuts (fast pipeline)"] = {"seconds": sum(stages.values()),
                               "result": f"R-hat {fast_diag['rhat'].max():.3f}, min ESS {fast_diag[['ess_bulk','ess_tail']].min().min():.0f}"}
display(pd.DataFrame(alt).T)

## 8. Cost, and the cluster

Wall time of each stage of the fast run on this machine (shared and loaded while it ran), and
the reference run's. The cluster estimate in the README scales these to MASSIVE: 8 chains on 8
cores per task.

In [ ]:
times = pd.DataFrame({
    "fast": json.loads((FAST / "stage_seconds.json").read_text()),
    "reference": json.loads((REFERENCE / "stage_seconds.json").read_text()),
}).fillna(0.0) / 60.0
times.T.plot.bar(title="Wall time by stage (minutes)", labels={"value": "minutes", "index": "run"}).show()
display(times.round(1))

## Notes

- **Likelihood: mean or sum over notification years.** The code that produced the published
  posterior (estival 0.6) averages each target's normal log-densities over its years; the
  paper's methods appendix (§9.2) writes a sum. Only notifications has several years (25), so
  the sum would weight that target 25 times more. The port keeps the code's mean (it is what
  the published posterior is); `pixi run calibrate --aggregate sum` runs the other.
- **Bound-limited parameters.** The marginals in section 4 show which parameters pile against
  a prior bound; the data push those against the priors, so their posteriors are partly the
  prior's choice.